# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [20]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

class CilentError(Exception):
  """400/404: our bug, never fall back."""
def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }

    last = None

    for attempt in range(tries):
        try:
            r = requests.get(BASE, params=params, timeout=15)

            if r.status_code == 200:
                return pd.DataFrame(r.json()['daily'])

            if r.status_code not in RETRY_STATUSES:
                raise RuntimeError(
                    f'not retryable: {r.status_code} {r.text[:200]}')

            last = f'status {r.status_code}'

        except requests.RequestException as e:
            last = repr(e)

        if attempt < tries - 1:
            delay = 0.5 * (2 ** attempt)
            print(f'attempt {attempt + 1} failed ({last}); retrying in {delay}s')
            time.sleep(delay)

    raise CilentError(
        f'not retryable: {r.status_code}{r.text[:200]}')

### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [21]:
used_fallback = False

try:
  weather=get_weather('2026-09-05', '2026-10-03')
  weather=weather[weather['time'].isin(['2026-09-05','2026-10-03'])].copy()
  assert len(weather)==2,'expected both dates'
  print(weather)

except RuntimeError as e:
  used_fallback=True
  print('Weather API failure:', e)
  print('Fallback data needed in Part 3')


          time  temperature_2m_max  precipitation_sum
0   2026-09-05                29.4                2.5
28  2026-10-03                22.0               14.7


### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [22]:
FALLBACK = pd.DataFrame({
    'time': ['2026-09-05', '2026-10-03'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})

if used_fallback:
  wx=FALLBACK.copy()
  wx['source']='fallback sample'
else:
  wx=weather.copy()
  wx['source']='open-meteo'

print(wx)

          time  temperature_2m_max  precipitation_sum      source
0   2026-09-05                29.4                2.5  open-meteo
28  2026-10-03                22.0               14.7  open-meteo


### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [23]:
wx=wx.rename(columns={
    'time':'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum':'precip_mm'
})

wx['date']=pd.to_datetime(wx['date'])
wx=wx[['date','temp_max','precip_mm', 'source']].reset_index(drop=True)

print(wx)

        date  temp_max  precip_mm      source
0 2026-09-05      29.4        2.5  open-meteo
1 2026-10-03      22.0       14.7  open-meteo


### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [24]:
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
assert pd.api.types.is_datetime64_any_dtype(wx['date'])
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'
print('weather table is safe to join.')

weather table is safe to join.


### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [25]:
wetter=wx.loc[wx['precip_mm'].idxmax()]
drier=wx.loc[wx['precip_mm'].idxmin()]
diff=wetter['precip_mm']- drier ['precip_mm']

if diff==0:
  print(f"Same rainfall on both days ({wetter['precip_mm']: .1f}mm).")
else:
  print(
    f"{wetter['date']:%B %d, %Y} was the wetter weekend, with"
    f"{wetter['precip_mm']:.1f} mm of rain versus {drier['precip_mm']:.1f}mm"
    f"on {drier['date']:%B %d, %Y}, a difference of {diff: .1f} mm."
  )

if (wx['source']=='fallback sample').any():
  print("Sample data, not real observations")

October 03, 2026 was the wetter weekend, with14.7 mm of rain versus 2.5mmon September 05, 2026, a difference of  12.2 mm.


### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [26]:
from datetime import datetime
LAT, LON= 38.03, -78.51
START, END='2026-09-05', '2026-10-03'

print('Endpoint:   ', BASE)
print('Coordinates:    ', f'{LAT,{LON}}')
print('Date range:   ', f'{START} to {END}')
print('Pulled:   ', datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
print('Data source:   ', 'FALLBACK SAMPLE (not real observations)' if used_fallback
      else 'real data from Open-Meteo')

Endpoint:    https://archive-api.open-meteo.com/v1/archive
Coordinates:     (38.03, {-78.51})
Date range:    2026-09-05 to 2026-10-03
Pulled:    2026-10-12 03:43:45
Data source:    real data from Open-Meteo


### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

If get_weather returned an empty DataFrame on failure instead of raising, there would be no obvious issues and the notebook would keep running. If I were to merge it onto 200 days of sales, every weather column would be NaN. The chart would be blank, or, if I filled the gaps with 0 to make it plot, it would show a flat line at zero rain. Either way, it would appear there is no relationship between rain and revenue.

Nobody would catch it because it is plausible that weather doesn't affect revenue. There would be no error pointing to the API call.

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [30]:
DATES = ['2026-09-05', '2026-10-03']

def clean_weather(raw, city):
    """Filter to the two dates and rename columns to our schema."""
    df = raw[raw['time'].isin(DATES)].copy()
    assert len(df) == 2, f'expected both dates for {city}'
    df = df.rename(columns={
        'time': 'date',
        'temperature_2m_max': 'temp_max',
        'precipitation_sum': 'precip_mm',
    })
    df['date'] = pd.to_datetime(df['date'])
    df['city'] = city
    df['source'] = 'open-meteo'
    return df[['city', 'date', 'temp_max', 'precip_mm', 'source']]

orlando = clean_weather(
    get_weather('2026-09-05', '2026-10-03', lat=28.54, lon=-81.38),
    'Orlando',
)

cville = wx.copy()
cville['city'] = 'Charlottesville'
cville = cville[['city', 'date', 'temp_max', 'precip_mm', 'source']]

weather_long = pd.concat([cville, orlando], ignore_index=True)
print(weather_long)

              city       date  temp_max  precip_mm      source
0  Charlottesville 2026-09-05      29.4        2.5  open-meteo
1  Charlottesville 2026-10-03      22.0       14.7  open-meteo
2          Orlando 2026-09-05      32.1        3.9  open-meteo
3          Orlando 2026-10-03      30.4       14.5  open-meteo
